# VELALAR COLLEGE OF ENGINEERING AND TECHNOLOGY (AUTONOMOUS), ERODE - 12
## DEPARTMENT OF INFORMATION TECHNOLOGY | ACADEMIC YEAR: 2026 - 2027 (ODD)
### Course: 25ITC31 - Foundations of Data Science
### Activity: Project-Based Assessment | Scenario 30: Movie Theatre Ticket Booking and Box Office Analysis System
**Register Numbers:** 30, 60, 90, 120

---
### Notebook Objectives:
1. **Unit 1:** Data Description & Descriptive Statistics (Mean, Median, Mode, Variance, Std Dev, Z-Scores)
2. **Unit 2:** Data Ingestion from Database/CSV File (`theatre_bookings.csv`)
3. **Unit 3:** Data Wrangling using NumPy and Pandas (Hierarchical Grouping & Pivot Tables)
4. **Unit 4:** Data Cleaning, Preprocessing, Outlier Treatment (IQR Method), and Feature Engineering
5. **Unit 5:** Exploratory Data Analysis (EDA) and Visualizations using Matplotlib and Seaborn

## 1. Importing Libraries & Dataset Ingestion (Unit 2)

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set display and visual configurations
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
sns.set_theme(style="whitegrid")

# Load dataset exported from MySQL database
df = pd.read_csv('theatre_bookings.csv')
print(f"Dataset Shape: {df.shape[0]} rows x {df.shape[1]} columns")
df.head()

## 2. Data Cleaning & Transformation Techniques (Unit 4)

In [2]:
# Audit missing values
print("Missing values per column:\n", df.isnull().sum())

# Impute refund amounts
df['refund_amount'] = df['refund_amount'].fillna(0.0)

# Remove duplicates if any
df.drop_duplicates(subset=['booking_id'], inplace=True)

# Data type conversions
df['booking_date'] = pd.to_datetime(df['booking_date'])
df['show_date'] = pd.to_datetime(df['show_date'])
df['ticket_price'] = df['ticket_price'].astype(float)
df['gross_amount'] = df['gross_amount'].astype(float)
df['refund_amount'] = df['refund_amount'].astype(float)

# Feature Engineering
df['net_amount'] = df['gross_amount'] - df['refund_amount']
df['booking_day_of_week'] = df['booking_date'].dt.day_name()
df['lead_time_days'] = (df['show_date'] - df['booking_date']).dt.total_seconds() / (24 * 3600)

# Outlier Detection using IQR on gross_amount
q1 = df['gross_amount'].quantile(0.25)
q3 = df['gross_amount'].quantile(0.75)
iqr = q3 - q1
lower_bound = q1 - 1.5 * iqr
upper_bound = q3 + 1.5 * iqr
print(f"IQR: {iqr:.2f} | Outlier bounds: [{lower_bound:.2f}, {upper_bound:.2f}]")

## 3. Descriptive Statistics & Z-Score Analysis (Unit 1)

In [3]:
numerical_cols = ['ticket_price', 'number_of_tickets', 'gross_amount', 'net_amount', 'show_occupancy_pct']
summary_stats = []

for col in numerical_cols:
    summary_stats.append({
        'Variable': col,
        'Mean': round(df[col].mean(), 2),
        'Median': round(df[col].median(), 2),
        'Mode': round(df[col].mode()[0], 2),
        'Std_Dev': round(df[col].std(), 2),
        'Variance': round(df[col].var(), 2),
        'Range': round(df[col].max() - df[col].min(), 2),
        'IQR': round(df[col].quantile(0.75) - df[col].quantile(0.25), 2)
    })

stats_df = pd.DataFrame(summary_stats)
print(stats_df)

# Z-Score for identifying high-revenue transactions
df['revenue_zscore'] = (df['gross_amount'] - df['gross_amount'].mean()) / df['gross_amount'].std()
df[['booking_id', 'customer_name', 'movie_title', 'gross_amount', 'revenue_zscore']].sort_values(by='revenue_zscore', ascending=False).head()

## 4. Data Wrangling & Pivot Tables (Unit 3)

In [4]:
# Movie-wise revenue and occupancy aggregation
movie_agg = df.groupby('movie_title').agg(
    Total_Bookings=('booking_id', 'count'),
    Tickets_Sold=('number_of_tickets', 'sum'),
    Gross_Revenue=('gross_amount', 'sum'),
    Total_Refunds=('refund_amount', 'sum'),
    Net_Revenue=('net_amount', 'sum'),
    Avg_Occupancy=('show_occupancy_pct', 'mean')
).reset_index().sort_values(by='Net_Revenue', ascending=False)

print(movie_agg)

# Pivot Table: Screen Name vs Genre Occupancy
pivot_tbl = df.pivot_table(
    index='screen_name',
    columns='genre',
    values='show_occupancy_pct',
    aggfunc='mean',
    fill_value=0.0
).round(2)
print("\nPivot Table:\n", pivot_tbl)

## 5. Visualizations & Exploratory Data Analysis (Unit 5)

In [5]:
# Figure 1: Movie Revenue Comparison (Gross vs Net)
plt.figure(figsize=(10, 5))
x = np.arange(len(movie_agg))
width = 0.35
plt.bar(x - width/2, movie_agg['Gross_Revenue'], width, label='Gross Revenue', color='#1f77b4')
plt.bar(x + width/2, movie_agg['Net_Revenue'], width, label='Net Revenue', color='#2ca02c')
plt.xticks(x, movie_agg['movie_title'], rotation=20, ha='right')
plt.ylabel('Revenue in ₹')
plt.title('Movie Box Office: Gross vs Net Revenue')
plt.legend()
plt.tight_layout()
plt.show()

In [6]:
# Figure 2: Revenue Contribution by Genre
genre_data = df.groupby('genre')['net_amount'].sum()
plt.figure(figsize=(7, 7))
plt.pie(genre_data, labels=genre_data.index, autopct='%1.1f%%', startangle=140)
plt.title('Net Revenue Contribution by Genre')
plt.show()

In [7]:
# Figure 3: Correlation Matrix Heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(df[['ticket_price', 'number_of_tickets', 'gross_amount', 'refund_amount', 'net_amount', 'show_occupancy_pct']].corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()